# Discrimination results — bar plots

Reads the `eval_trajectory*.json` files produced by `run_evaluation_discrimination.py`
and makes bar charts. Run top-to-bottom.

- Edit `GLOB` below if your results live elsewhere.
- Multiple json files for the same (source animal, prompt) are merged automatically.
- Figures display inline **and** are saved to `outputs/discrim/plots/`.


In [3]:
import json, re, glob
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

GLOB = "outputs/discrim/*/eval_trajectory*.json"   # <-- edit if needed
PLOTS = Path("outputs/discrim/plots"); PLOTS.mkdir(parents=True, exist_ok=True)

## Load & merge results

In [4]:
def parse_file(fp):
    d = json.loads(Path(fp).read_text())
    m = re.search(r"/([a-z]+)_vs_control", str(Path(fp)).replace("\\", "/"))
    return {"source": m.group(1) if m else "?",
            "prompt": d.get("system_prompt") is not None,
            "results": d.get("results", {})}

def step_of(l):
    if l == "base": return 0
    if l.startswith("checkpoint-"): return int(l.split("-")[-1])
    return None

def final_label(res):
    if "final" in res: return "final"
    cks = [l for l in res if l.startswith("checkpoint-")]
    return max(cks, key=step_of) if cks else "base"

def test_animal(source, ts):
    if ts == "indist": return source
    if ts.startswith("transfer_"): return ts[len("transfer_"):]
    return ts

def test_sets_of(res):
    seen = []
    for row in res.values():
        for ts in row:
            if ts not in seen: seen.append(ts)
    return seen

def cond_label(c):
    return f"{c['source']}-trained" + (" +prompt" if c["prompt"] else "")

files = sorted(glob.glob(GLOB))
assert files, f"no files match {GLOB}"
merged = {}
for fp in files:
    c = parse_file(fp)
    if not c["results"]: continue
    slot = merged.setdefault((c["source"], c["prompt"]),
                             {"source": c["source"], "prompt": c["prompt"], "results": {}})
    for label, row in c["results"].items():
        slot["results"].setdefault(label, {}).update(row)
conds = list(merged.values())
print("files:", len(files))
print("conditions:", [cond_label(c) for c in conds])

AssertionError: no files match outputs/discrim/*/eval_trajectory*.json

## Text summary (base → final per test set)

In [ ]:
for c in conds:
    res = c["results"]; fl = final_label(res)
    base, fin = res.get("base", {}), res.get(fl, {})
    print(f"\n== {cond_label(c)} ==")
    for ts in test_sets_of(res):
        b = base.get(ts, {}).get("auroc", float("nan"))
        f = fin.get(ts, {}).get("auroc", float("nan"))
        kind = "in-dist " if ts == "indist" else "transfer"
        print(f"  {kind} {test_animal(c['source'], ts):<8} base={b:.3f}  final={f:.3f}  delta={f-b:+.3f}")

## Bar 1 — Final AUROC by test animal, grouped by detector (no-prompt)
The transfer story: how each detector scores on each animal. In-dist bars should be highest within a detector; high off-animal bars = generalization.

In [ ]:
np_conds = [c for c in conds if not c["prompt"]]
tests = sorted({test_animal(c["source"], ts) for c in np_conds for ts in test_sets_of(c["results"])})
x = np.arange(len(tests)); w = 0.8 / max(len(np_conds), 1)
plt.figure(figsize=(1.7 * len(tests) + 3, 4.5))
for i, c in enumerate(np_conds):
    fin = c["results"].get(final_label(c["results"]), {})
    vals = [fin.get("indist" if t == c["source"] else f"transfer_{t}", {}).get("auroc", np.nan) for t in tests]
    plt.bar(x + i * w - 0.4 + w / 2, vals, w, label=cond_label(c))
plt.axhline(0.5, ls="--", c="gray", lw=1)
plt.xticks(x, tests); plt.ylim(0.45, 0.95)
plt.xlabel("tested on (animal vs control)"); plt.ylabel("final AUROC")
plt.title("Final detector AUROC by test animal"); plt.legend(); plt.grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.savefig(PLOTS / "bar_by_test_animal.png", dpi=130); plt.show()

## Bar 2 — Base vs Final per condition
What training added on top of the untrained model (delta = generalizable signal learned).

In [ ]:
for c in conds:
    res = c["results"]; fl = final_label(res)
    base, fin = res.get("base", {}), res.get(fl, {})
    tsets = test_sets_of(res); labels = [test_animal(c["source"], ts) for ts in tsets]
    b = [base.get(ts, {}).get("auroc", np.nan) for ts in tsets]
    f = [fin.get(ts, {}).get("auroc", np.nan) for ts in tsets]
    x = np.arange(len(tsets))
    plt.figure(figsize=(1.6 * len(tsets) + 3, 4.5))
    plt.bar(x - 0.2, b, 0.4, label="base (untrained)")
    plt.bar(x + 0.2, f, 0.4, label="final (trained)")
    plt.axhline(0.5, ls="--", c="gray", lw=1)
    plt.xticks(x, labels); plt.ylim(0.45, 0.95); plt.ylabel("AUROC")
    plt.title(f"{cond_label(c)}: base vs final"); plt.legend(); plt.grid(alpha=0.3, axis="y")
    plt.tight_layout(); plt.savefig(PLOTS / f"bar_base_final_{c['source']}{'_prompt' if c['prompt'] else ''}.png", dpi=130); plt.show()

## Bar 3 — With vs without the "love" system prompt
Only drawn for sources that have both runs. Tests whether being an animal-lover helps detection.

In [ ]:
by_source = {}
for c in conds:
    by_source.setdefault(c["source"], {})[c["prompt"]] = c
drew = False
for source, v in by_source.items():
    if not (True in v and False in v): continue
    drew = True
    no, yes = v[False], v[True]
    tsets = test_sets_of(no["results"]); labels = [test_animal(source, ts) for ts in tsets]
    nf = [no["results"][final_label(no["results"])].get(ts, {}).get("auroc", np.nan) for ts in tsets]
    yf = [yes["results"][final_label(yes["results"])].get(ts, {}).get("auroc", np.nan) for ts in tsets]
    x = np.arange(len(tsets))
    plt.figure(figsize=(1.6 * len(tsets) + 3, 4.5))
    plt.bar(x - 0.2, nf, 0.4, label="no prompt")
    plt.bar(x + 0.2, yf, 0.4, label='+"love" prompt')
    plt.axhline(0.5, ls="--", c="gray", lw=1)
    plt.xticks(x, labels); plt.ylim(0.45, 0.95); plt.ylabel("final AUROC")
    plt.title(f"{source}-trained: prompt vs no-prompt"); plt.legend(); plt.grid(alpha=0.3, axis="y")
    plt.tight_layout(); plt.savefig(PLOTS / f"bar_prompt_{source}.png", dpi=130); plt.show()
if not drew:
    print("No source has both a with-prompt and without-prompt run yet.")